<a href="https://colab.research.google.com/github/meryambutt123-a11y/urdu-ocr-codesaviours-si26-maryam/blob/main/SI26-Week3-Maryam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3: Dataset Expansion & PyTorch Dataset Class

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 1. Install missing libraries
!pip install -q arabic-reshaper python-bidi

# 2. Download the RAW TrueType font file directly from Google Fonts
!wget -q "https://raw.githubusercontent.com/googlefonts/noto-fonts/main/unhinted/ttf/NotoNastaliqUrdu/NotoNastaliqUrdu-Regular.ttf" -O font.ttf

# 3. Generate images and save directly to Drive
import os
import arabic_reshaper
from bidi.algorithm import get_display
from PIL import Image, ImageDraw, ImageFont

# Update "Urdu_OCR_Project" if your main folder is named differently
drive_path = "/content/drive/MyDrive/Urdu_OCR_Project/data/raw/synthetic"
os.makedirs(drive_path, exist_ok=True)

font_path = "font.ttf"
font_size = 60
font = ImageFont.truetype(font_path, font_size)

sentences = [
    "کوڈنگ کرتے کرتے بال سفید ہو گئے۔",
    "میرے لیپ ٹاپ کی بیٹری ختم ہونے والی ہے۔",
    "آج پھر سے ایرر آ گیا۔",
    "ڈیٹا سیٹ بنانا سب سے مشکل کام ہے۔",
    "ایک اور کافی کا کپ چاہیے۔",
    "پروجیکٹ کی ڈیڈ لائن کل ہے۔",
    "میرا کوڈ کام کیوں نہیں کر رہا؟",
    "مشین لرننگ نے دماغ گھما دیا ہے۔",
    "ویک تھری بہت لمبا لگ رہا ہے۔",
    "آخر کار کوڈ چل ہی گیا۔"
]

for i, text in enumerate(sentences, start=1):
    img = Image.new('RGB', (900, 150), color='white')
    draw = ImageDraw.Draw(img)

    bidi_text = get_display(arabic_reshaper.reshape(text))
    text_bbox = draw.textbbox((0, 0), bidi_text, font=font)

    x = (900 - (text_bbox[2] - text_bbox[0])) / 2
    y = (150 - (text_bbox[3] - text_bbox[1])) / 2

    draw.text((x, y), bidi_text, font=font, fill='black')

    img.save(f"{drive_path}/synthetic_batch2_{i}.jpg")

print(f"✅ Success! 10 images saved to: {drive_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 8.9 MB/s eta 0:00:00
✅ Success! 10 images saved to: /content/drive/MyDrive/Urdu_OCR_Project/data/raw/synthetic


In [3]:
import os
import glob

# Your base directory
base_dir = "/content/drive/MyDrive/Urdu_OCR_Project/data/raw"

folders = {
    "books": {"start_index": 6, "prefix": "book_"},
    "newspaper": {"start_index": 1, "prefix": "newspaper_"}, # Assuming you don't have previous newspaper files
    "signboards": {"start_index": 1, "prefix": "signboard_"}  # Assuming you don't have previous signboard files
}

for folder_name, config in folders.items():
    folder_path = os.path.join(base_dir, folder_name)

    # Find all generic image files (screenshots or downloaded images)
    # Adjust the extension if your images are .jpg instead of .png
    messy_files = glob.glob(os.path.join(folder_path, "*.*"))
    messy_files.sort()

    current_index = config["start_index"]

    for old_path in messy_files:
        # Skip if it's already properly named
        if config["prefix"] in os.path.basename(old_path):
            continue

        extension = os.path.splitext(old_path)[1]
        new_name = f"{config['prefix']}{current_index}{extension}"
        new_path = os.path.join(folder_path, new_name)

        try:
            os.rename(old_path, new_path)
            print(f"Renamed in {folder_name}: {os.path.basename(old_path)} -> {new_name}")
            current_index += 1
        except Exception as e:
            print(f"Error renaming {old_path}: {e}")

print("✅ Bulk renaming complete!")

✅ Bulk renaming complete!


In [4]:
import os
import glob
import re

folder_path = "/content/drive/MyDrive/Urdu_OCR_Project/data/raw/other"
all_files = glob.glob(os.path.join(folder_path, "*.*"))

max_index = 0
messy_files = []


for file_path in all_files:
    filename = os.path.basename(file_path)
    if filename.startswith("others_"):
        match = re.search(r'\d+', filename)
        if match:
            max_index = max(max_index, int(match.group()))
    else:
        messy_files.append(file_path)

current_index = max_index + 1
messy_files.sort()

for old_path in messy_files:
    extension = os.path.splitext(old_path)[1]
    new_name = f"others_{current_index}{extension}"
    new_path = os.path.join(folder_path, new_name)

    os.rename(old_path, new_path)
    print(f"✅ Renamed: {os.path.basename(old_path)} -> {new_name}")
    current_index += 1

In [5]:
import os
import glob

folders = ['books', 'newspaper', 'signboards', 'synthetic', 'other']
base_dir = "/content/drive/MyDrive/Urdu_OCR_Project/data/raw"

total_images = 0
print("📂 Current Physical Image Count in Google Drive:\n" + "-"*40)

for folder in folders:
    folder_path = os.path.join(base_dir, folder)
    # Count all files in this specific folder
    files = glob.glob(os.path.join(folder_path, "*.*"))
    count = len(files)
    total_images += count
    print(f"{folder.capitalize():<12}: {count} images")

print("-" * 40)
print(f"Total physically in Drive: {total_images} images")

📂 Current Physical Image Count in Google Drive:
----------------------------------------
Books       : 50 images
Newspaper   : 35 images
Signboards  : 41 images
Synthetic   : 40 images
Other       : 44 images
----------------------------------------
Total physically in Drive: 210 images


In [6]:
import os
import glob
import re

base_dir = "/content/drive/MyDrive/Urdu_OCR_Project/data/raw"

# Define the exact prefix for each folder
folders = {
    "books": "book_",
    "newspaper": "newspaper_",
    "signboards": "signboard_"
}

for folder_name, prefix in folders.items():
    folder_path = os.path.join(base_dir, folder_name)
    if not os.path.exists(folder_path):
        continue

    all_files = glob.glob(os.path.join(folder_path, "*.*"))

    max_index = 0
    messy_files = []

    # Separate the nicely named files from the messy new ones
    for file_path in all_files:
        filename = os.path.basename(file_path)

        if filename.startswith(prefix):
            # Extract the number from the existing valid file
            match = re.search(r'\d+', filename)
            if match:
                num = int(match.group())
                if num > max_index:
                    max_index = num
        else:
            # It doesn't have the correct prefix, so it's a new messy file
            messy_files.append(file_path)

    messy_files.sort()

    # Start the next index strictly after your highest existing file
    current_index = max_index + 1

    for old_path in messy_files:
        extension = os.path.splitext(old_path)[1]
        new_name = f"{prefix}{current_index}{extension}"
        new_path = os.path.join(folder_path, new_name)

        try:
            os.rename(old_path, new_path)
            print(f"✅ Renamed in {folder_name}: {os.path.basename(old_path)}  ->  {new_name}")
            current_index += 1
        except Exception as e:
            print(f"❌ Error renaming {old_path}: {e}")

print("\n🚀 All files auto-sequenced and renamed successfully!")


🚀 All files auto-sequenced and renamed successfully!


In [7]:
import os
import glob

csv_path = "/content/drive/MyDrive/Urdu_OCR_Project/data/labels.csv"
raw_paths = glob.glob('/content/drive/MyDrive/Urdu_OCR_Project/data/raw/*/*.*')

# Read existing paths in the CSV so we don't create duplicates
existing_paths = set()
if os.path.exists(csv_path):
    with open(csv_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split(',', 1)
            existing_paths.add(parts[0].strip())

# Append the missing files with a comma, leaving the text part blank
with open(csv_path, 'a', encoding='utf-8') as f:
    added_count = 0
    for path in sorted(raw_paths):
        # Format the path exactly how your PyTorch dataset expects it
        short_path = path.split('Urdu_OCR_Project/')[1] if 'Urdu_OCR_Project/' in path else path

        if short_path not in existing_paths:
            f.write(f"{short_path},\n")
            added_count += 1

print(f"✅ Successfully added {added_count} new file paths to labels.csv!")

✅ Successfully added 10 new file paths to labels.csv!


In [8]:
import os
import glob

old_csv_path = "/content/drive/MyDrive/Urdu_OCR_Project/data/labels.csv"
new_csv_path = "/content/drive/MyDrive/Urdu_OCR_Project/data/labels_v2.csv"
raw_paths = glob.glob('/content/drive/MyDrive/Urdu_OCR_Project/data/raw/*/*.*')

# 1. Save all your hard work from the old file
existing_records = {}
if os.path.exists(old_csv_path):
    with open(old_csv_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split(',', 1)
            if len(parts) == 2:
                existing_records[parts[0].strip()] = parts[1].strip()

# 2. Write everything cleanly into a brand new file
with open(new_csv_path, 'w', encoding='utf-8') as f:
    added_count = 0
    for path in sorted(raw_paths):
        # Format the path exactly how your PyTorch dataset expects it
        short_path = path.split('Urdu_OCR_Project/')[1] if 'Urdu_OCR_Project/' in path else path

        # If you already typed the text, keep it. Otherwise, leave blank.
        text = existing_records.get(short_path, "")
        f.write(f"{short_path},{text}\n")
        added_count += 1

print(f"✅ Success! Created a fresh 'labels_v2.csv' with exactly {added_count} rows.")

✅ Success! Created a fresh 'labels_v2.csv' with exactly 210 rows.


In [9]:
# Download a robust Naskh font that natively supports reshaped characters
!wget -q -O urdu_font.ttf https://raw.githubusercontent.com/googlefonts/noto-fonts/main/unhinted/ttf/NotoNaskhArabic/NotoNaskhArabic-Regular.ttf

import os
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display

out_dir = "/content/drive/MyDrive/Urdu_OCR_Project/data/raw/synthetic"

urdu_texts = [
    "یہ میری پہلی کتاب ہے",
    "آج موسم بہت اچھا ہے",
    "پاکستان ایک خوبصورت ملک ہے",
    "علم بڑی دولت ہے",
    "محنت میں عظمت ہے",
    "وقت کی قدر کرو",
    "ہمیشہ سچ بولو",
    "صاف پانی پیو",
    "درخت لگانا صدقہ ہے",
    "یہ ایک نیا دن ہے"
]

# Configure the reshaper specifically for Urdu
reshaper = arabic_reshaper.ArabicReshaper(configuration={'language': 'Urdu'})

# Load the new stable font
font = ImageFont.truetype("urdu_font.ttf", 60)

for i, text in enumerate(urdu_texts, start=1):
    reshaped_text = reshaper.reshape(text)
    bidi_text = get_display(reshaped_text)

    img = Image.new('RGB', (900, 150), color='white')
    draw = ImageDraw.Draw(img)

    # Standard centering math
    bbox = draw.textbbox((0, 0), bidi_text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    x = (900 - text_w) / 2
    y = (150 - text_h) / 2 - bbox[1]

    draw.text((x, y), bidi_text, font=font, fill='black')

    file_path = os.path.join(out_dir, f"synthetic_fixed_{i}.jpg")
    img.save(file_path)

print("✅ Successfully generated 10 Naskh images (No more boxes!)")

✅ Successfully generated 10 Naskh images (No more boxes!)


In [10]:
# Re-download the stable Naskh font directly into the fresh Colab session
!wget -q -O urdu_font.ttf https://raw.githubusercontent.com/googlefonts/noto-fonts/main/unhinted/ttf/NotoNaskhArabic/NotoNaskhArabic-Regular.ttf

import os
from PIL import Image, ImageDraw, ImageFont

out_dir = "/content/drive/MyDrive/Urdu_OCR_Project/data/raw/synthetic"

urdu_texts = [
    "یہ میری پہلی کتاب ہے",
    "آج موسم بہت اچھا ہے",
    "پاکستان ایک خوبصورت ملک ہے",
    "علم بڑی دولت ہے",
    "محنت میں عظمت ہے",
    "وقت کی قدر کرو",
    "ہمیشہ سچ بولو",
    "صاف پانی پیو",
    "درخت لگانا صدقہ ہے",
    "یہ ایک نیا دن ہے"
]

# Load the stable Naskh font we just downloaded
font = ImageFont.truetype("urdu_font.ttf", 60)

for i, text in enumerate(urdu_texts, start=1):
    img = Image.new('RGB', (900, 150), color='white')
    draw = ImageDraw.Draw(img)

    # Let Pillow naturally calculate the bounding box for Right-to-Left Urdu text
    bbox = draw.textbbox((0, 0), text, font=font, direction='rtl', language='ur')
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    # Standard centering math
    x = (900 - text_w) / 2
    y = (150 - text_h) / 2 - bbox[1]

    # Draw the text naturally with Pillow's built-in RTL support
    draw.text((x, y), text, font=font, fill='black', direction='rtl', language='ur')

    file_path = os.path.join(out_dir, f"synthetic_fixed_{i}.jpg")
    img.save(file_path)

print("✅ Clean, connected, Right-to-Left Urdu generated perfectly!")

✅ Clean, connected, Right-to-Left Urdu generated perfectly!


In [11]:
import os
import glob

# Delete all old broken 'batch2' images
old_files = glob.glob("/content/drive/MyDrive/Urdu_OCR_Project/data/raw/synthetic/synthetic_batch2_*.jpg")
for f in old_files:
    os.remove(f)

# Print the correct new filenames to open
new_files = glob.glob("/content/drive/MyDrive/Urdu_OCR_Project/data/raw/synthetic/synthetic_fixed_*.jpg")
print("Open these exact files in Google Drive:")
for f in sorted(new_files):
    print(os.path.basename(f))

Open these exact files in Google Drive:
synthetic_fixed_1.jpg
synthetic_fixed_10.jpg
synthetic_fixed_2.jpg
synthetic_fixed_3.jpg
synthetic_fixed_4.jpg
synthetic_fixed_5.jpg
synthetic_fixed_6.jpg
synthetic_fixed_7.jpg
synthetic_fixed_8.jpg
synthetic_fixed_9.jpg


In [12]:
import sentencepiece
print(sentencepiece.__version__)

0.2.2


In [16]:
!pip install "transformers==4.57.1" "sentencepiece==0.2.0" --force-reinstall

  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached sentencepiece-0.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.7 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [14]:
from transformers.utils import is_sentencepiece_available, is_tiktoken_available
print("sentencepiece available:", is_sentencepiece_available())
print("tiktoken available:", is_tiktoken_available())

import transformers, tokenizers
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)

# also test the actual protobuf-backed module sentencepiece uses internally —
# this is the piece that silently breaks with newer protobuf even though
# `import sentencepiece` alone succeeds
from sentencepiece import sentencepiece_model_pb2
print("sentencepiece_model_pb2 loaded OK")

import google.protobuf
print("protobuf:", google.protobuf.__version__)

sentencepiece available: True
tiktoken available: True
transformers: 5.13.1
tokenizers: 0.22.2
sentencepiece_model_pb2 loaded OK
protobuf: 5.29.6


In [1]:
import transformers
print(transformers.__version__)

4.57.1


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [7]:
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class UrduOCRDataset(Dataset):
    def __init__(self, csv_file, root_dir, processor, max_length=128):
        self.df = pd.read_csv(csv_file, header=None, names=['file_path', 'text'])
        self.df = self.df.dropna().reset_index(drop=True)
        self.root_dir = root_dir
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_path = self.df.iloc[idx]['file_path']
        text = str(self.df.iloc[idx]['text'])
        img_path = os.path.join(self.root_dir, file_path)
        image = Image.open(img_path).convert("RGB")

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_length,
            truncation=True
        ).input_ids

        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

dataset = UrduOCRDataset(
    csv_file="/content/drive/MyDrive/Urdu_OCR_Project/data/labels.csv",
    root_dir="/content/drive/MyDrive/Urdu_OCR_Project/",
    processor=processor
)

print(f"Total valid samples loaded: {len(dataset)}")
sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Total valid samples loaded: 200
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 160
Testing samples: 40
